# TuringBench DeBERTa-v3-large Fine-tuning (Colab)

Fine-tune `microsoft/deberta-v3-large` on TuringBench using Google Colab.

## Before you run
1. Set `HF_TOKEN` via Colab Secrets (key icon on the left) if you want to push to HuggingFace Hub.
2. If no token is set, the model will still train and be saved to `/content/models/`.


In [ ]:
# Install dependencies compatible with Colab T4 (CUDA 12.2)
!pip install -q --upgrade transformers datasets accelerate scikit-learn pandas joblib huggingface-hub

In [ ]:
import os
from google.colab import userdata

# Try to read HF_TOKEN from Colab Secrets.
try:
    os.environ['HF_TOKEN'] = userdata.get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets')
except Exception as e:
    print('HF_TOKEN not set. Training will continue, but Hub push will be disabled.')
    print('Set it via the key icon -> Secrets -> HF_TOKEN')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/content/ai-detection-at-scale'
if not os.path.exists(repo_dir):
    !git clone $repo_url $repo_dir
else:
    !git -C $repo_dir pull origin main
print('Repo ready at', repo_dir)

In [ ]:
script = os.path.join(repo_dir, 'scripts', '33_finetune_turingbench.py')
out_dir = '/content/models/turingbench_deberta_v3_large'
os.makedirs(out_dir, exist_ok=True)

hub_model_id = 'vedangvatsa123/vedang-turingbench-deberta-v3-large'

# Resume from the latest Hub checkpoint if one exists and HF_TOKEN is set.
resume_from = None
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import list_repo_refs
        refs = list(list_repo_refs(repo_id=hub_model_id).branches)
        if refs:
            resume_from = hub_model_id
            print('Resuming from Hub checkpoint:', resume_from)
    except Exception as e:
        print('No existing Hub checkpoint found, starting fresh:', e)
else:
    print('HF_TOKEN not set; checkpoint resume disabled.')

# Colab T4 needs the same FP32 / small-batch config as Kaggle P100.
cmd = (
    f"python {script} "
    f"--model_name microsoft/deberta-v3-large "
    f"--output_dir {out_dir} "
    f"--max_length 256 "
    f"--epochs 1 "
    f"--batch_size 8 "
    f"--gradient_accumulation_steps 4 "
    f"--learning_rate 1e-5 "
    f"--no_fp16 "
    f"--no_class_weights "
    f"--hub_model_id {hub_model_id} "
)
if resume_from:
    cmd += f"--resume_from_checkpoint {resume_from} "
cmd += "--seed 42"

print('Running:', cmd)
!{cmd}

In [ ]:
# Optional: save model to Google Drive if Hub push was not used.
import shutil
from google.colab import drive

if not os.environ.get('HF_TOKEN'):
    drive.mount('/content/drive')
    drive_out = '/content/drive/MyDrive/turingbench_deberta_v3_large'
    shutil.copytree(out_dir, drive_out, dirs_exist_ok=True)
    print(f'Model copied to Google Drive: {drive_out}')
else:
    print('Model was pushed to HuggingFace Hub.')